# Causal vs Predictive Features: The Distribution Shift Test

## What You Will Do
You will build a dataset where causal and spurious-predictive features are explicitly separated. You will select features using SHAP (predictive importance) and a conditional independence test (a proxy for causal relevance). Then you will apply distribution shift — changing the environment so spurious correlations break — and measure how badly predictive-only selection degrades while causal selection holds up.

## The Core Insight
**Predictive features** are correlated with the target in the training distribution. They may have no direct causal relationship — they work because the training environment happened to pair them with the target.

**Causal features** directly influence the target through a stable mechanism. Under distribution shift — a different time period, a different market regime, a different population — spurious correlations break but causal mechanisms survive.

This is the "aha moment" for feature selection: the best training-set features are often the worst deployment features.

## Estimated Time: under 2 minutes

---

## Setup

In [ ]:
# Purpose: Import all dependencies
# Key Concept: We use conditional independence testing (partial correlation) as a
#              lightweight causal discovery tool — not a full causal graph, but directional

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from scipy.stats import pearsonr
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

## Build the Dataset with Known Causal Structure

We generate data where we know the ground truth: which features cause the target, and which are spuriously correlated through a **confounder**.

The causal graph:

```
x_causal_1 ──┐
              ├──> y (target)
x_causal_2 ──┘

confounder ───> x_spurious_1    confounder also influences y through a path
confounder ───> x_spurious_2    that is severed under distribution shift
confounder ─┐
            └──> y (this path breaks under shift)
```

In training: `x_spurious_1` and `x_spurious_2` appear predictive because they share the confounder with the target.
Under shift: the confounder–target relationship is severed; spurious features stop working.

In [ ]:
# Purpose: Generate a synthetic dataset with explicit causal and spurious features
# Key Concept: We engineer the data-generating process so we know exactly which
#              features are causal — this is ground truth that real data rarely provides

def make_causal_dataset(n_train=600, n_test_clean=100, n_test_shift=300, random_state=42):
    """
    Generate a classification dataset with causal and spurious features.

    The confounder drives both spurious features AND the target in training.
    Under distribution shift, the confounder-target path is severed.

    Returns
    -------
    X_train, y_train, X_test_clean, y_test_clean, X_test_shift, y_test_shift
    feature_names : list of str (with labels causal/spurious)
    causal_idx : list of int
    spurious_idx : list of int
    """
    rng = np.random.RandomState(random_state)
    N = n_train + n_test_clean + n_test_shift

    # Causal features: directly influence y, independent of environment
    x1 = rng.randn(N)  # causal
    x2 = rng.randn(N)  # causal

    # Confounder: drives spurious features AND the target in training
    confounder = rng.randn(N)

    # Spurious features: correlated with y only through confounder
    xs1 = 0.8 * confounder + 0.2 * rng.randn(N)
    xs2 = 0.6 * confounder + 0.4 * rng.randn(N)

    # Target: driven by causal features + confounder (in training environment)
    logit = 0.9 * x1 + 0.7 * x2 + 0.9 * confounder + 0.3 * rng.randn(N)
    prob = 1 / (1 + np.exp(-logit))
    y = (prob > 0.5).astype(int)

    # Full feature matrix
    X_all = np.column_stack([x1, x2, xs1, xs2])
    feature_names = ['x_causal_1', 'x_causal_2', 'x_spurious_1', 'x_spurious_2']
    causal_idx = [0, 1]
    spurious_idx = [2, 3]

    # Train / clean test split (same distribution)
    X_train = X_all[:n_train]
    y_train = y[:n_train]
    X_test_clean = X_all[n_train:n_train + n_test_clean]
    y_test_clean = y[n_train:n_train + n_test_clean]

    # Shift test: confounder-target relationship reverses; spurious features flip
    # The causal features still drive y through the same mechanism
    shift_start = n_train + n_test_clean
    x1_shift = X_all[shift_start:, 0]
    x2_shift = X_all[shift_start:, 1]
    confounder_shift = -confounder[shift_start:]  # FLIP: confounder relationship reverses

    xs1_shift = 0.8 * confounder_shift + 0.2 * rng.randn(n_test_shift)
    xs2_shift = 0.6 * confounder_shift + 0.4 * rng.randn(n_test_shift)

    # y is still driven by the causal features (mechanism unchanged)
    # confounder is no longer part of the y mechanism (structural break)
    logit_shift = 0.9 * x1_shift + 0.7 * x2_shift + 0.3 * rng.randn(n_test_shift)
    prob_shift = 1 / (1 + np.exp(-logit_shift))
    y_shift = (prob_shift > 0.5).astype(int)

    X_test_shift = np.column_stack([x1_shift, x2_shift, xs1_shift, xs2_shift])

    return (X_train, y_train, X_test_clean, y_test_clean,
            X_test_shift, y_shift, feature_names, causal_idx, spurious_idx)


(X_train, y_train, X_test_clean, y_test_clean,
 X_test_shift, y_test_shift,
 feature_names, causal_idx, spurious_idx) = make_causal_dataset()

print(f"Training samples   : {len(X_train)}")
print(f"Clean test samples : {len(X_test_clean)}")
print(f"Shift test samples : {len(X_test_shift)}")
print(f"Features           : {feature_names}")
print(f"Causal features    : {[feature_names[i] for i in causal_idx]}")
print(f"Spurious features  : {[feature_names[i] for i in spurious_idx]}")

## Verify the Spurious Correlations

Before selection, confirm that the spurious features look useful on training data — they correlate with the target — but their correlation flips or disappears under shift. This is the trap.

In [ ]:
# Purpose: Show that spurious features appear predictive in training but not under shift
# Key Concept: Correlation is not causation — and here we can measure exactly how much
#              the spurious correlation changes when the environment shifts

print("Pearson correlation with target (y):")
print(f"{'Feature':<20} {'Train corr':>12} {'Clean test corr':>16} {'Shift test corr':>16}")
print('-' * 68)

for i, name in enumerate(feature_names):
    r_train, _ = pearsonr(X_train[:, i], y_train)
    r_clean, _ = pearsonr(X_test_clean[:, i], y_test_clean)
    r_shift, _ = pearsonr(X_test_shift[:, i], y_test_shift)
    tag = '  <- causal' if i in causal_idx else '  <- SPURIOUS'
    print(f"{name:<20} {r_train:>12.4f} {r_clean:>16.4f} {r_shift:>16.4f}{tag}")

print()
print("Note: spurious features show high training correlation but low/reversed shift correlation.")
print("Causal features maintain stable correlation across environments.")

## Feature Selection Method 1 — SHAP (Predictive)

SHAP measures how much each feature contributes to model predictions — in the **training distribution**. If spurious features are correlated with the target in training, SHAP will correctly report that the model uses them. But this accuracy is conditional on the training environment holding.

In [ ]:
# Purpose: Select features using SHAP importance on the training set
# Key Concept: SHAP faithfully reflects what the model learned from training data;
#              it does not distinguish causal from spurious if both are correlated in training

scaler = StandardScaler()
Xs_train = scaler.fit_transform(X_train)
Xs_clean = scaler.transform(X_test_clean)
Xs_shift = scaler.transform(X_test_shift)

# Train a Random Forest on all features
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xs_train, y_train)

# SHAP values from TreeExplainer
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(Xs_train)
shap_arr = np.array(shap_values)

# Mean absolute SHAP: average contribution magnitude per feature
if shap_arr.ndim == 3:
    mean_abs_shap = np.abs(shap_arr).mean(axis=0).mean(axis=1)
else:
    mean_abs_shap = np.abs(shap_arr).mean(axis=0)

print("SHAP importance scores (training set):")
shap_ranking = np.argsort(mean_abs_shap)[::-1]
for rank, idx in enumerate(shap_ranking, start=1):
    tag = 'CAUSAL' if idx in causal_idx else 'SPURIOUS'
    print(f"  {rank}. {feature_names[idx]:<20}  mean|SHAP| = {mean_abs_shap[idx]:.4f}  [{tag}]")

# Select top 2 by SHAP
shap_top2_idx = shap_ranking[:2]
shap_selected = [feature_names[i] for i in shap_top2_idx]
print(f"\nSHAP selects top 2: {shap_selected}")

## Feature Selection Method 2 — Conditional Independence Test (Causal Proxy)

A feature is causally relevant if it is conditionally associated with the target even after controlling for other features. We use **partial correlation**: the correlation between a feature and the target after removing the linear effect of all other features. Spurious features that only work through the confounder will lose their partial correlation when the confounder is controlled.

This is a lightweight causal proxy — not a full causal discovery algorithm, but it is enough to demonstrate the principle.

In [ ]:
# Purpose: Compute partial correlations as a causal proxy for feature selection
# Key Concept: Partial correlation measures direct association after controlling for
#              shared confounders; spurious features lose significance under this test

def partial_correlation(X_arr, y_arr, feature_idx, control_idx):
    """
    Compute partial correlation of feature_idx with y_arr,
    controlling for control_idx features via linear regression residuals.

    Parameters
    ----------
    X_arr : ndarray (n_samples, n_features)
    y_arr : ndarray (n_samples,)
    feature_idx : int — feature to test
    control_idx : list of int — features to control for

    Returns
    -------
    r : float — partial correlation coefficient
    p : float — p-value
    """
    if len(control_idx) == 0:
        return pearsonr(X_arr[:, feature_idx], y_arr)

    lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=500)

    # Residualise the feature: remove variance explained by controls
    from sklearn.linear_model import LinearRegression
    X_controls = X_arr[:, control_idx]
    lm_feat = LinearRegression().fit(X_controls, X_arr[:, feature_idx])
    resid_feat = X_arr[:, feature_idx] - lm_feat.predict(X_controls)

    # Residualise the target (treat as continuous for partial correlation)
    lm_y = LinearRegression().fit(X_controls, y_arr.astype(float))
    resid_y = y_arr.astype(float) - lm_y.predict(X_controls)

    r, p = pearsonr(resid_feat, resid_y)
    return r, p


ALPHA = 0.05  # significance level
causal_selected = []

print("Partial correlation test (controlling for all other features):")
print(f"{'Feature':<20} {'Partial r':>10} {'p-value':>10} {'Significant':>12} {'True label'}")
print('-' * 70)

for i, name in enumerate(feature_names):
    controls = [j for j in range(len(feature_names)) if j != i]
    r, p = partial_correlation(Xs_train, y_train, i, controls)
    significant = p < ALPHA
    if significant:
        causal_selected.append(name)
    true_label = 'CAUSAL' if i in causal_idx else 'SPURIOUS'
    print(f"{name:<20} {r:>10.4f} {p:>10.4f} {str(significant):>12} {true_label}")

causal_test_idx = [feature_names.index(f) for f in causal_selected]
print(f"\nConditional independence test selects: {causal_selected}")

## The Distribution Shift Test

Now the moment of truth. We train models on each feature set and evaluate them on three test sets:
1. **Clean test**: same distribution as training
2. **Shift test**: confounder-target relationship reversed

If causal selection works, the gap between clean and shift accuracy will be smaller for the causal model than for the predictive model.

In [ ]:
# Purpose: Compare causal vs predictive selection under distribution shift
# Key Concept: The accuracy gap between clean and shift test is the "robustness tax"
#              paid for using spurious features

lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=500)

def evaluate_subset(feature_idx_list, label):
    """Train on training set, evaluate on clean and shift test sets."""
    idx = list(feature_idx_list)
    if len(idx) == 0:
        return None

    lr_model.fit(Xs_train[:, idx], y_train)
    acc_train = accuracy_score(y_train, lr_model.predict(Xs_train[:, idx]))
    acc_clean = accuracy_score(y_test_clean, lr_model.predict(Xs_clean[:, idx]))
    acc_shift = accuracy_score(y_test_shift, lr_model.predict(Xs_shift[:, idx]))
    degradation = acc_clean - acc_shift

    print(f"{label}")
    print(f"  Features  : {[feature_names[i] for i in idx]}")
    print(f"  Train acc : {acc_train:.4f}")
    print(f"  Clean acc : {acc_clean:.4f}")
    print(f"  Shift acc : {acc_shift:.4f}")
    print(f"  Degradation (clean -> shift): {degradation:+.4f}")
    print()
    return {'label': label, 'train': acc_train, 'clean': acc_clean,
            'shift': acc_shift, 'degradation': degradation, 'idx': idx}

results = {}

# All features
results['all'] = evaluate_subset(range(4), 'ALL FEATURES')

# SHAP selection (predictive)
results['shap'] = evaluate_subset(shap_top2_idx, f'SHAP TOP-2 (predictive): {shap_selected}')

# Conditional independence (causal proxy)
if len(causal_test_idx) > 0:
    results['causal'] = evaluate_subset(causal_test_idx,
        f'CONDITIONAL INDEPENDENCE (causal proxy): {causal_selected}')

# True causal features (oracle — cheating)
results['oracle'] = evaluate_subset(causal_idx, 'ORACLE CAUSAL (ground truth)')

## Visualisation — The Robustness Gap

In [ ]:
# Purpose: Visualise the accuracy degradation under distribution shift for each method
# Key Concept: The width of the gap between clean and shift bars is the robustness cost
#              of using spurious features

plot_items = [(k, v) for k, v in results.items() if v is not None]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Grouped bar chart — clean vs shift accuracy per method
ax = axes[0]
methods = [r['label'].split(' (')[0][:20] for _, r in plot_items]
clean_accs = [r['clean'] for _, r in plot_items]
shift_accs = [r['shift'] for _, r in plot_items]

x = np.arange(len(methods))
width = 0.35
bars1 = ax.bar(x - width/2, clean_accs, width, label='Clean test (same dist)', color='#2ca02c', alpha=0.85)
bars2 = ax.bar(x + width/2, shift_accs, width, label='Shift test (distribution shift)', color='#d62728', alpha=0.85)

# Draw arrows showing degradation
for xi, (c, s) in zip(x, zip(clean_accs, shift_accs)):
    if c > s + 0.005:
        ax.annotate('', xy=(xi + width/2, s + 0.005), xytext=(xi - width/2, c - 0.005),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Accuracy Under Distribution Shift\n(arrows show degradation direction)', fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.grid(axis='y', alpha=0.3)

# Right: Degradation bar chart
ax2 = axes[1]
degradations = [r['degradation'] for _, r in plot_items]
colors_deg = ['#d62728' if d > 0.05 else '#ff7f0e' if d > 0.01 else '#2ca02c'
              for d in degradations]
bars3 = ax2.bar(methods, degradations, color=colors_deg, alpha=0.85, edgecolor='white')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_ylabel('Accuracy degradation (clean - shift)', fontsize=10)
ax2.set_title('Robustness Cost of Each Selection Strategy\n'
              '(red = large degradation, green = robust)', fontsize=11)
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels(methods, rotation=15, ha='right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# Value labels
for bar, deg in zip(bars3, degradations):
    ax2.text(bar.get_x() + bar.get_width()/2,
             deg + (0.003 if deg >= 0 else -0.015),
             f'{deg:+.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Why SHAP Cannot Detect Spurious Features on Training Data

This cell makes the limitation concrete: we show the SHAP values for spurious and causal features, demonstrating that in training, SHAP cannot distinguish them.

In [ ]:
# Purpose: Demonstrate that SHAP correctly reflects model behaviour in training,
#          even when the model uses spurious features that will fail under shift
# Key Concept: SHAP is a faithful explanation tool — the issue is with what the
#              model learned, not with SHAP itself

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shap_arr = np.array(shap_values)
if shap_arr.ndim == 3:
    shap_for_plot = shap_arr[:, :, 1]  # class 1
else:
    shap_for_plot = shap_arr

# Left: SHAP values per feature (beeswarm style)
ax = axes[0]
colors_feat = ['#2ca02c', '#2ca02c', '#d62728', '#d62728']  # green=causal, red=spurious
for row_pos, (feat_name, col_color) in enumerate(zip(feature_names, colors_feat)):
    shap_col = shap_for_plot[:, row_pos]
    feat_vals = Xs_train[:, row_pos]
    feat_norm = (feat_vals - feat_vals.min()) / (feat_vals.ptp() + 1e-9)
    ax.scatter(
        shap_col,
        np.full_like(shap_col, row_pos) + np.random.uniform(-0.15, 0.15, len(shap_col)),
        c=feat_norm, cmap='coolwarm', alpha=0.4, s=10, linewidths=0
    )

ax.set_yticks(range(len(feature_names)))
labels_with_type = [
    f"{name} [{'CAUSAL' if i in causal_idx else 'SPURIOUS'}]"
    for i, name in enumerate(feature_names)
]
ax.set_yticklabels(labels_with_type, fontsize=10)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value (positive = pushes toward class 1)', fontsize=10)
ax.set_title('SHAP Values on Training Distribution\n'
             '(SHAP cannot distinguish causal from spurious\n'
             ' if both are correlated with target in training)', fontsize=10)
ax.grid(axis='x', alpha=0.3)

# Right: Feature correlations with target across environments
ax2 = axes[1]
environments = ['Training', 'Clean Test', 'Shift Test']
Xs_environments = [Xs_train, Xs_clean, Xs_shift]
y_environments = [y_train, y_test_clean, y_test_shift]

x_env = np.arange(len(environments))
width = 0.2

for feat_i, (feat_name, col_color) in enumerate(zip(feature_names, colors_feat)):
    corrs = [
        pearsonr(Xe[:, feat_i], ye)[0]
        for Xe, ye in zip(Xs_environments, y_environments)
    ]
    offset = (feat_i - 1.5) * width
    style = '-o' if feat_i in causal_idx else '--s'
    ax2.plot(x_env, corrs, style, color=col_color, label=feat_name, linewidth=2, markersize=8)

ax2.axhline(y=0, color='grey', linestyle=':', alpha=0.5)
ax2.set_xticks(x_env)
ax2.set_xticklabels(environments, fontsize=11)
ax2.set_ylabel('Pearson correlation with target', fontsize=10)
ax2.set_title('Feature-Target Correlation Across Environments\n'
              '(spurious correlations flip; causal correlations hold)', fontsize=10)
ax2.legend(fontsize=9, loc='lower left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise: Increase the Spurious Correlation Strength

**Task:** Regenerate the dataset with a stronger confounder effect on the spurious features (increase the coefficient from 0.8 to 1.5). Observe how this makes the SHAP-based selection degrade more severely under shift, while the conditional independence test still identifies the correct features.

**Expected outcome:** Stronger spurious correlation means SHAP is more deceived in training, and the accuracy gap under shift widens.

<details>
<summary>Hint 1</summary>
Copy the make_causal_dataset function, change the xs1 coefficient from 0.8 to a larger value (1.5, 2.0), then re-run the correlation table and SHAP analysis.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
A stronger confounder coefficient means xs1 = 1.5 * confounder + ... — the spurious feature becomes more tightly linked to the confounder, making it more predictive in training and more wrong under shift.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

SPURIOUS_STRENGTH = 1.5  # Modify this: try 1.0, 1.5, 2.0

def make_causal_dataset_tunable(spurious_strength=0.8, n_train=600, n_test_shift=300,
                                 random_state=42):
    """Same as make_causal_dataset but with tunable spurious feature strength."""
    rng = np.random.RandomState(random_state)
    N = n_train + 100 + n_test_shift

    x1 = rng.randn(N)
    x2 = rng.randn(N)
    confounder = rng.randn(N)

    # Tunable spurious strength
    noise_strength = max(0.05, 1.0 - spurious_strength * 0.5)
    xs1 = spurious_strength * confounder + noise_strength * rng.randn(N)
    xs2 = spurious_strength * 0.75 * confounder + noise_strength * rng.randn(N)

    logit = 0.9 * x1 + 0.7 * x2 + 0.9 * confounder + 0.3 * rng.randn(N)
    y = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)

    X_all = np.column_stack([x1, x2, xs1, xs2])

    X_train = X_all[:n_train]
    y_train = y[:n_train]

    shift_start = n_train + 100
    x1_s = X_all[shift_start:, 0]
    x2_s = X_all[shift_start:, 1]
    conf_s = -confounder[shift_start:]
    xs1_s = spurious_strength * conf_s + noise_strength * rng.randn(n_test_shift)
    xs2_s = spurious_strength * 0.75 * conf_s + noise_strength * rng.randn(n_test_shift)
    logit_s = 0.9 * x1_s + 0.7 * x2_s + 0.3 * rng.randn(n_test_shift)
    y_shift = (1 / (1 + np.exp(-logit_s)) > 0.5).astype(int)
    X_shift = np.column_stack([x1_s, x2_s, xs1_s, xs2_s])

    return X_train, y_train, X_shift, y_shift


Xt_new, yt_new, Xs_new, ys_new = make_causal_dataset_tunable(
    spurious_strength=SPURIOUS_STRENGTH
)
sc_new = StandardScaler()
Xt_new_s = sc_new.fit_transform(Xt_new)
Xs_new_s = sc_new.transform(Xs_new)

# Train a model on all features and check shift degradation
lr_new = LogisticRegression(random_state=RANDOM_STATE, max_iter=500)
lr_new.fit(Xt_new_s, yt_new)

acc_train_new = accuracy_score(yt_new, lr_new.predict(Xt_new_s))
acc_shift_new = accuracy_score(ys_new, lr_new.predict(Xs_new_s))
degradation_new = acc_train_new - acc_shift_new

print(f"Spurious strength: {SPURIOUS_STRENGTH}")
print(f"Train accuracy   : {acc_train_new:.4f}")
print(f"Shift accuracy   : {acc_shift_new:.4f}")
print(f"Degradation      : {degradation_new:+.4f}")

your_degradation = degradation_new

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise():
    assert your_degradation is not None, "Run the tunable dataset experiment above."
    assert isinstance(your_degradation, float), "your_degradation should be a float."
    assert SPURIOUS_STRENGTH >= 1.0, (
        f"Set SPURIOUS_STRENGTH to at least 1.0 (currently {SPURIOUS_STRENGTH}) "
        "to observe meaningful degradation."
    )
    # Stronger spurious correlations should cause more degradation
    assert your_degradation > 0.0, (
        f"Degradation {your_degradation:.4f} should be positive when spurious "
        "features are present and distribution shifts. Check the shift test setup."
    )
    print(f"Exercise passed! Spurious strength = {SPURIOUS_STRENGTH}, "
          f"shift degradation = {your_degradation:+.4f}.")
    print("The stronger the spurious correlation, the larger the degradation under shift.")

test_exercise()

## Summary

**Key Takeaways:**

1. **Predictive features** are correlated with the target in the training distribution — SHAP correctly identifies them. But correlation can be spurious, driven by a confounder that changes across environments.
2. **Causal features** are directly mechanistically linked to the target. Their relationship with the target is invariant across environments because the mechanism does not change.
3. **Distribution shift is the test**: the performance gap between clean and shift test is the price paid for using spurious features.
4. **Conditional independence testing** (partial correlation) provides a lightweight causal proxy that filters out confounder-mediated associations.
5. **SHAP is not broken** — it faithfully reflects what the model learned. The limitation is in using training-distribution importance as a universal quality measure.

**Practical rule of thumb:** if your model will be deployed in a different environment from where it was trained (different time period, different market, different population), prefer selection methods that probe causal stability over those that only measure training-set importance.

**What is next:**
- Module 9 (Causal Feature Selection) for invariant causal prediction (ICP) and anchor regression
- Module 7 (Time Series) for the time-series equivalent: non-stationary feature importance
- `feature_selection_in_5_minutes.ipynb` if you want to revisit the three core methods